Major Transformations

In [1]:
from pyspark.sql import functions as F
from pyspark.sql import Window

StatementMeta(, b2a1e5a4-5974-4727-badb-4bfbfb7d4d0a, 3, Finished, Available, Finished, False)

In [2]:
silver = 'abfss://RBusiness360_Fabric_Project@onelake.dfs.fabric.microsoft.com/RBusiness360_LakeHouse.Lakehouse/Files/Silver'
gold = 'abfss://RBusiness360_Fabric_Project@onelake.dfs.fabric.microsoft.com/RBusiness360_LakeHouse.Lakehouse/Files/Gold'

StatementMeta(, b2a1e5a4-5974-4727-badb-4bfbfb7d4d0a, 4, Finished, Available, Finished, False)

In [3]:
def read_silver(table: str):
    return spark.read.parquet(f"{silver}/{table}")

def write_gold(df, table_name: str):
    path = f"{gold}/{table_name}"
    (
        df.write
        .mode("overwrite")
        .parquet(path)
    )
    print(f"[gold] wrote {table_name} → {path}  ({df.count()} rows)")

StatementMeta(, b2a1e5a4-5974-4727-badb-4bfbfb7d4d0a, 5, Finished, Available, Finished, False)

In [5]:
customers     = read_silver("Customers")
products      = read_silver("Products")
sales         = read_silver("Transactions")
inventory     = read_silver("Inventory")
interactions  = read_silver("Interactions")
stores        = read_silver("Stores")
shipments     = read_silver("Shipments")

StatementMeta(, b2a1e5a4-5974-4727-badb-4bfbfb7d4d0a, 7, Finished, Available, Finished, False)

In [6]:
interaction_summary = (
    interactions.groupBy("customer_id")
    .agg(
        F.count("interaction_id").alias("total_interactions"),
        F.max("interaction_date").alias("last_interaction_date"),
        F.sum(F.when(F.col("interaction_type") == "Complaint", 1).otherwise(0))
         .alias("complaint_count"),
    )
)

dim_customers = (
    customers
    .drop("_silver_load_ts")
    .join(interaction_summary, on="customer_id", how="left")
    .withColumn("total_interactions", F.coalesce(F.col("total_interactions"), F.lit(0)))
    .withColumn("complaint_count",    F.coalesce(F.col("complaint_count"),    F.lit(0)))
    .withColumn("_gold_load_ts", F.current_timestamp())
)

write_gold(dim_customers, "dim_customers")

dim_products = (
    products
    .drop("_silver_load_ts")
    .withColumn("_gold_load_ts", F.current_timestamp())
)

write_gold(dim_products, "dim_products")

dim_stores = (
    stores
    .drop("_silver_load_ts")
    .withColumn("_gold_load_ts", F.current_timestamp())
)

write_gold(dim_stores, "dim_stores")


StatementMeta(, b2a1e5a4-5974-4727-badb-4bfbfb7d4d0a, 8, Finished, Available, Finished, False)

[gold] wrote dim_customers → abfss://RBusiness360_Fabric_Project@onelake.dfs.fabric.microsoft.com/RBusiness360_LakeHouse.Lakehouse/Files/Gold/dim_customers  (10000 rows)
[gold] wrote dim_products → abfss://RBusiness360_Fabric_Project@onelake.dfs.fabric.microsoft.com/RBusiness360_LakeHouse.Lakehouse/Files/Gold/dim_products  (2000 rows)
[gold] wrote dim_stores → abfss://RBusiness360_Fabric_Project@onelake.dfs.fabric.microsoft.com/RBusiness360_LakeHouse.Lakehouse/Files/Gold/dim_stores  (30 rows)


In [7]:

fact_sales = (
    sales
    .drop("_silver_load_ts")
    .join(
        products.select("product_id", "product_name", "category", "brand", "unit_price"),
        on="product_id", how="left"
    )
    .join(
        customers.select("customer_id", "full_name", "city", "country",
                         "loyalty_tier", "age", "gender"),
        on="customer_id", how="left"
    )
    .join(
        stores.select("store_id", "store_name", "region"),
        on="store_id", how="left"
    )

    .withColumn("expected_amount",   F.col("quantity") * F.col("unit_price"))
    .withColumn("discount_amount",
        F.greatest(F.col("expected_amount") - F.col("sales_amount"), F.lit(0.0))
    )
    .withColumn("discount_pct",
        F.when(F.col("expected_amount") > 0,
               F.round(F.col("discount_amount") / F.col("expected_amount") * 100, 2)
        ).otherwise(0.0)
    )
    .withColumn("year",    F.year("transaction_date"))
    .withColumn("month",   F.month("transaction_date"))
    .withColumn("quarter", F.quarter("transaction_date"))
    .withColumn("week",    F.weekofyear("transaction_date"))
    .withColumn("_gold_load_ts", F.current_timestamp())
)

write_gold(fact_sales, "fact_sales")

StatementMeta(, b2a1e5a4-5974-4727-badb-4bfbfb7d4d0a, 9, Finished, Available, Finished, False)

[gold] wrote fact_sales → abfss://RBusiness360_Fabric_Project@onelake.dfs.fabric.microsoft.com/RBusiness360_LakeHouse.Lakehouse/Files/Gold/fact_sales  (85 rows)


In [8]:
agg_sales_by_day = (
    fact_sales
    .groupBy("transaction_date", "year", "month", "quarter", "week")
    .agg(
        F.countDistinct("transaction_id").alias("transaction_count"),
        F.sum("sales_amount").alias("total_revenue"),
        F.sum("quantity").alias("units_sold"),
        F.avg("sales_amount").alias("avg_order_value"),
        F.sum("discount_amount").alias("total_discounts"),
    )
    .withColumn("total_revenue",   F.round("total_revenue",   2))
    .withColumn("avg_order_value", F.round("avg_order_value", 2))
    .withColumn("total_discounts", F.round("total_discounts", 2))
    .orderBy("transaction_date")
    .withColumn("_gold_load_ts", F.current_timestamp())
)

write_gold(agg_sales_by_day, "agg_sales_by_day")

agg_sales_by_product = (
    fact_sales
    .groupBy("product_id", "product_name", "category", "brand")
    .agg(
        F.sum("sales_amount").alias("total_revenue"),
        F.sum("quantity").alias("units_sold"),
        F.countDistinct("transaction_id").alias("transaction_count"),
        F.countDistinct("customer_id").alias("unique_customers"),
        F.avg("sales_amount").alias("avg_order_value"),
        F.avg("discount_pct").alias("avg_discount_pct"),
    )
    .withColumn("total_revenue",   F.round("total_revenue",   2))
    .withColumn("avg_order_value", F.round("avg_order_value", 2))
    .withColumn("avg_discount_pct",F.round("avg_discount_pct",2))
    # rank products by revenue
    .withColumn("revenue_rank",
        F.rank().over(Window.orderBy(F.desc("total_revenue")))
    )
    .withColumn("_gold_load_ts", F.current_timestamp())
)

write_gold(agg_sales_by_product, "agg_sales_by_product")


agg_sales_by_customer = (
    fact_sales
    .groupBy("customer_id", "full_name", "city", "country",
             "loyalty_tier", "age", "gender")
    .agg(
        F.sum("sales_amount").alias("lifetime_revenue"),
        F.sum("quantity").alias("total_units_purchased"),
        F.countDistinct("transaction_id").alias("total_orders"),
        F.min("transaction_date").alias("first_purchase_date"),
        F.max("transaction_date").alias("last_purchase_date"),
        F.avg("sales_amount").alias("avg_order_value"),
        F.countDistinct("product_id").alias("distinct_products_bought"),
    )
    .withColumn("lifetime_revenue",   F.round("lifetime_revenue",   2))
    .withColumn("avg_order_value",    F.round("avg_order_value",    2))
    
    .withColumn("days_since_last_purchase",
        F.datediff(F.current_date(), F.col("last_purchase_date"))
    )
    
    .withColumn("rfm_segment",
        F.when(F.col("lifetime_revenue") > 5000, "Champion")
        .when(F.col("lifetime_revenue") > 2000, "Loyal")
        .when(F.col("lifetime_revenue") > 500,  "Potential")
        .otherwise("New")
    )
    .withColumn("_gold_load_ts", F.current_timestamp())
)

write_gold(agg_sales_by_customer, "agg_sales_by_customer")


agg_sales_by_region = (
    fact_sales
    .groupBy("region", "year", "quarter")
    .agg(
        F.sum("sales_amount").alias("total_revenue"),
        F.sum("quantity").alias("units_sold"),
        F.countDistinct("transaction_id").alias("transaction_count"),
        F.countDistinct("customer_id").alias("unique_customers"),
        F.avg("sales_amount").alias("avg_order_value"),
    )
    .withColumn("total_revenue",   F.round("total_revenue",   2))
    .withColumn("avg_order_value", F.round("avg_order_value", 2))
    .orderBy("region", "year", "quarter")
    .withColumn("_gold_load_ts", F.current_timestamp())
)

write_gold(agg_sales_by_region, "agg_sales_by_region")


latest_inventory = (
    inventory
    .withColumn("row_num",
        F.row_number().over(
            Window.partitionBy("warehouse_id", "product_id")
                  .orderBy(F.desc("inventory_date"))
        )
    )
    .filter(F.col("row_num") == 1)
    .drop("row_num", "_silver_load_ts")
)

agg_inventory_health = (
    latest_inventory
    .join(
        products.select("product_id", "product_name", "category", "unit_price"),
        on="product_id", how="left"
    )
    # cumulative stock on hand
    .withColumn("stock_on_hand", F.col("stock_received") - F.col("stock_dispatched"))
    # reorder threshold: flag if stock on hand < 10 % of received
    .withColumn("reorder_flag",
        F.when(
            F.col("stock_on_hand") < (F.col("stock_received") * 0.10), True
        ).otherwise(False)
    )
    .withColumn("stock_value",
        F.round(F.col("stock_on_hand") * F.col("unit_price"), 2)
    )
    .withColumn("_gold_load_ts", F.current_timestamp())
)

write_gold(agg_inventory_health, "agg_inventory_health")



agg_shipment_kpis = (
    shipments
    .groupBy("delivery_partner", "status")
    .agg(
        F.count("shipment_id").alias("shipment_count"),
        F.avg("delivery_days").alias("avg_delivery_days"),
        F.min("delivery_days").alias("min_delivery_days"),
        F.max("delivery_days").alias("max_delivery_days"),
        F.avg("shipping_cost").alias("avg_shipping_cost"),
        F.sum("shipping_cost").alias("total_shipping_cost"),
        F.avg("weight_kg").alias("avg_weight_kg"),
    )
    .withColumn("avg_delivery_days",   F.round("avg_delivery_days",   1))
    .withColumn("avg_shipping_cost",   F.round("avg_shipping_cost",   2))
    .withColumn("total_shipping_cost", F.round("total_shipping_cost", 2))
    .withColumn("avg_weight_kg",       F.round("avg_weight_kg",       2))
    # on-time delivery approximation: delivered in <= 7 days
    .withColumn("_gold_load_ts", F.current_timestamp())
)

write_gold(agg_shipment_kpis, "agg_shipment_kpis")

StatementMeta(, b2a1e5a4-5974-4727-badb-4bfbfb7d4d0a, 10, Finished, Available, Finished, False)

[gold] wrote agg_sales_by_day → abfss://RBusiness360_Fabric_Project@onelake.dfs.fabric.microsoft.com/RBusiness360_LakeHouse.Lakehouse/Files/Gold/agg_sales_by_day  (84 rows)
[gold] wrote agg_sales_by_product → abfss://RBusiness360_Fabric_Project@onelake.dfs.fabric.microsoft.com/RBusiness360_LakeHouse.Lakehouse/Files/Gold/agg_sales_by_product  (84 rows)
[gold] wrote agg_sales_by_customer → abfss://RBusiness360_Fabric_Project@onelake.dfs.fabric.microsoft.com/RBusiness360_LakeHouse.Lakehouse/Files/Gold/agg_sales_by_customer  (85 rows)
[gold] wrote agg_sales_by_region → abfss://RBusiness360_Fabric_Project@onelake.dfs.fabric.microsoft.com/RBusiness360_LakeHouse.Lakehouse/Files/Gold/agg_sales_by_region  (56 rows)
[gold] wrote agg_inventory_health → abfss://RBusiness360_Fabric_Project@onelake.dfs.fabric.microsoft.com/RBusiness360_LakeHouse.Lakehouse/Files/Gold/agg_inventory_health  (28684 rows)
[gold] wrote agg_shipment_kpis → abfss://RBusiness360_Fabric_Project@onelake.dfs.fabric.microsoft.co